In [ ]:
import pandas as pd
import numpy as np
import re

# ĐÃ SỬA LỖI: Cập nhật đường dẫn lùi 2 bước (../../) để chỉ đúng vào thư mục data của nhóm
train_df = pd.read_csv('../../data/processed/train_v2.csv')
test_df = pd.read_csv('../../data/processed/test_v2.csv')

# Hàm loại bỏ các ký tự đặc biệt JSON trong tên cột để tránh lỗi LightGBM
def sanitize_column_names(df):
    # Thay thế các ký tự [ ], {, }, :, , và " thành dấu gạch dưới _
    df.columns = [re.sub(r'[\[\]\,\{\}\:\"]', '_', col) for col in df.columns]
    return df

# Tiến hành làm sạch tên cột cho cả 2 tập dữ liệu
train_df = sanitize_column_names(train_df)
test_df = sanitize_column_names(test_df)

# Xác định cột Target là 'Price'
target_col = 'Price'

X_train = train_df.drop(columns=[target_col])
y_train = train_df[target_col]

X_test = test_df.drop(columns=[target_col])
y_test = test_df[target_col]

# Khôi phục giá trị thực tế của Price về đơn vị gốc bằng expm1 (vì trước đó dùng log transform)
y_test_actual = np.expm1(y_test)

print("Kích thước tập Train:", X_train.shape)
print("Kích thước tập Test:", X_test.shape)
print("Đã tải dữ liệu thành công và xử lý xong tên cột!")

In [ ]:
import pandas as pd
import numpy as np

# Giả sử bạn đã load X_train, X_test từ file train_v2.csv và test_v2.csv
# Tạo biến mới: Diện tích trung bình cho mỗi phòng ngủ
# Cộng 1 để tránh lỗi chia cho 0 đối với các nhà studio không có phòng ngủ riêng
X_train['Area_per_Bedroom'] = X_train['Area'] / (X_train['Bedrooms'] + 1)
X_test['Area_per_Bedroom'] = X_test['Area'] / (X_test['Bedrooms'] + 1)

# Tạo biến phân loại (Flag): Bất động sản diện tích lớn (>150m2)
X_train['Is_Large_House'] = (X_train['Area'] >= 150).astype(int)
X_test['Is_Large_House'] = (X_test['Area'] >= 150).astype(int)

print(f"Số lượng đặc trưng sau khi Feature Engineering: {X_train.shape[1]}")

In [ ]:
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

# Tách tập Validation từ tập Train để phục vụ Early Stopping
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

# --- 2.1 CẤU HÌNH XGBOOST VỚI EARLY STOPPING ---
print("Đang huấn luyện XGBoost...")
xgb_tuned = XGBRegressor(
    n_estimators=2000,        # Cho phép học tối đa 2000 cây
    learning_rate=0.03,       # Tốc độ học chậm để chính xác hơn
    max_depth=6,              # Độ sâu của cây
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

# Kích hoạt Early Stopping (Dùng fit thay vì cross_val)
xgb_tuned.fit(
    X_tr, y_tr,
    eval_set=[(X_val, y_val)],
    early_stopping_rounds=50, # Dừng nếu sau 50 cây không cải thiện
    verbose=False
)
print(f"XGBoost dừng lại ở cây thứ: {xgb_tuned.best_iteration}")

# --- 2.2 CẤU HÌNH LIGHTGBM VỚI EARLY STOPPING ---
print("Đang huấn luyện LightGBM...")
lgbm_tuned = LGBMRegressor(
    n_estimators=2000,
    learning_rate=0.03,
    num_leaves=31,
    random_state=42,
    verbose=-1
)

# Cách dùng early stopping cho LightGBM phiên bản mới qua callbacks
from lightgbm import early_stopping
lgbm_tuned.fit(
    X_tr, y_tr,
    eval_set=[(X_val, y_val)],
    eval_metric='rmse',
    callbacks=[early_stopping(stopping_rounds=50, verbose=False)]
)

In [ ]:
from sklearn.ensemble import StackingRegressor, RandomForestRegressor
from sklearn.linear_model import RidgeCV

# Chuẩn bị Random Forest đã tinh chỉnh cơ bản
rf_tuned = RandomForestRegressor(n_estimators=300, max_depth=15, random_state=42)

# Khai báo các mô hình Lớp 1 (Base Estimators)
estimators = [
    ('XGBoost', xgb_tuned),
    ('LightGBM', lgbm_tuned),
    ('RandomForest', rf_tuned)
]

# Xây dựng Stacking Regressor
print("Đang huấn luyện mô hình Stacking (Sẽ mất vài phút)...")
stacking_model = StackingRegressor(
    estimators=estimators,
    final_estimator=RidgeCV(), # Meta-model tự động tìm alpha tốt nhất
    cv=5,                      # Sử dụng K-Fold 5 lần bên trong Stacking
    n_jobs=-1                  # Chạy đa luồng cho nhanh
)

# Huấn luyện Stacking trên toàn bộ dữ liệu X_train gốc
stacking_model.fit(X_train, y_train)
print("Hoàn tất huấn luyện Stacking!")

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error, r2_score

# Dự báo trên tập Test
log_pred = stacking_model.predict(X_test)

# Chống tràn số (Clipping)
clipped_log_pred = np.clip(log_pred, a_min=0, a_max=25)

# Biến đổi ngược từ Logarit về giá tiền thật (Triệu VNĐ)
actual_pred = np.expm1(clipped_log_pred)
# Đảm bảo y_test cũng đã được biến đổi ngược
# actual_y_test = np.expm1(y_test) 

# Tính toán các chỉ số
mae = mean_absolute_error(actual_y_test, actual_pred)
rmse = np.sqrt(mean_squared_error(actual_y_test, actual_pred))
mape = mean_absolute_percentage_error(actual_y_test, actual_pred) * 100
r2 = r2_score(actual_y_test, actual_pred)
median_err = np.median(np.abs(actual_y_test - actual_pred))

print(f"--- HIỆU NĂNG MÔ HÌNH STACKING ---")
print(f"MAE          : {mae:,.2f} (Triệu VNĐ)")
print(f"RMSE         : {rmse:,.2f} (Triệu VNĐ)")
print(f"Median Error : {median_err:,.2f} (Triệu VNĐ)")
print(f"MAPE         : {mape:.2f}% (Tỷ lệ sai số trung bình)")
print(f"R² Score     : {r2:.4f}")

In [ ]:
from sklearn.inspection import permutation_importance
import matplotlib.pyplot as plt
import seaborn as sns

print("Đang tính toán Permutation Importance (Có thể mất 1-2 phút)...")
# Đo lường tầm quan trọng bằng cách xáo trộn dữ liệu trên tập Test 5 lần
perm_importance = permutation_importance(
    stacking_model, X_test, y_test, 
    n_repeats=5, random_state=42, 
    scoring='neg_mean_absolute_error'
)

# Trích xuất kết quả và sắp xếp
sorted_idx = perm_importance.importances_mean.argsort()[-15:] # Lấy Top 15 đặc trưng mạnh nhất

# Trực quan hóa bằng Boxplot
plt.figure(figsize=(12, 8))
sns.boxplot(
    data=perm_importance.importances[sorted_idx].T,
    orient='h',
    palette="viridis"
)
plt.yticks(range(len(sorted_idx)), X_test.columns[sorted_idx])
plt.xlabel("Mức độ sụt giảm hiệu năng (Nếu xáo trộn đặc trưng)", fontsize=12)
plt.title("Top 15 Feature Importance (Đo bằng Permutation Importance)", fontsize=14, fontweight='bold')
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()